# Loan Portfolio Risk Copilot — Feature Engineering

Preprocesses Lending Club loan data for a **behavioral** PD model — i.e. a model
scoring loans already on the book, as opposed to an application/underwriting
model scoring new applicants. See project README for the full scope discussion.

## Feature selection principles

Two categories of feature are used:

- **Origination features** — borrower and loan attributes fixed at issuance.
  Still necessary as a baseline risk signal even in a behavioral model.
- **Behavioral features** — attributes reflecting activity during the loan's
  life, distinct from its final outcome.

Several candidate fields were evaluated and excluded as leakage:

| Excluded field(s) | Reason |
|---|---|
| `total_pymnt`, `total_rec_prncp`, `total_rec_int`, `total_rec_late_fee` | Cumulative payments received — near-direct function of whether the loan defaulted |
| `out_prncp`, `out_prncp_inv` | Remaining principal — differs systematically by outcome |
| `recoveries`, `collection_recovery_fee` | Only nonzero for defaulted loans by definition |
| `debt_settlement_flag` | Entering settlement is almost always a *consequence* of default, not a lead indicator |
| `last_pymnt_d`, `last_pymnt_amnt`, `next_pymnt_d` | Entangled with terminal payment status |
| `last_fico_range_low/high`, `last_credit_pull_d`, and anything derived from them | The "last" credit pull is anchored to when a loan's outcome became final (confirmed via timing analysis: median pull for charged-off loans lands 15 months before scheduled maturity, vs. 2 months for fully-paid loans) — it encodes default timing, not a signal that precedes it |

The remaining behavioral features (`pymnt_plan`, `hardship_flag`) reflect
genuine in-life events without being definitionally tied to the outcome.

In [1]:
import pandas as pd
import numpy as np

## 1. Feature selection

In [3]:
# Origination / application-time features: borrower profile, credit bureau
# snapshot at application, and loan terms fixed at issuance.
origination_features = [
    'annual_inc', 'verification_status', 'emp_length', 'home_ownership',
    'addr_state', 'dti', 'purpose', 'emp_title',
    'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq',
    'mths_since_last_record', 'open_acc', 'total_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'delinq_2yrs', 'acc_now_delinq', 'num_tl_90g_dpd_24m',
    'tot_coll_amt', 'tot_cur_bal', 'mo_sin_old_rev_tl_op', 'mort_acc',
    'pct_tl_nvr_dlq', 'pub_rec_bankruptcies', 'earliest_cr_line',
    'loan_amnt', 'int_rate', 'term', 'installment', 'grade', 'sub_grade', 'issue_d',
]

# Behavioral features: reflect events during the loan's life without being
# definitionally tied to its outcome (see exclusion table above for what
# was evaluated and rejected, and why).
behavioral_features = [
    'pymnt_plan',    # currently on a formal payment plan
    'hardship_flag', # ever entered a hardship program
]

status_and_label = ['loan_status']

model_features = origination_features + behavioral_features + status_and_label

## 2. Load data

In [4]:
model_df = pd.read_csv(
    "../Data/accepted_2007_to_2018Q4.csv",
    low_memory=False,
    usecols=model_features
)

model_df.shape

(2260701, 38)

## 3. Label engineering

In [5]:
# Only resolved loans have a known outcome. Active loans ('Current', 'Late',
# 'In Grace Period') are held out entirely -- they seed the live-portfolio
# demo dataset instead (see 02_modeling_evaluation.ipynb, Section: Demo Portfolio).
keep_statuses = [
    'Fully Paid', 'Charged Off', 'Default',
    'Does not meet the credit policy. Status: Fully Paid',
    'Does not meet the credit policy. Status: Charged Off'
]
model_df = model_df[model_df['loan_status'].isin(keep_statuses)]

default_statuses = ['Charged Off', 'Default', 'Does not meet the credit policy. Status: Charged Off']
model_df['loan_status'] = model_df['loan_status'].apply(lambda x: 1 if x in default_statuses else 0)

model_df['loan_status'].value_counts(normalize=True)

loan_status
0    0.80035
1    0.19965
Name: proportion, dtype: float64

In [6]:
model_df['home_ownership'] = model_df['home_ownership'].replace(
    {'ANY': 'OTHER', 'NONE': 'OTHER', 'OTHER': 'OTHER'}
)

## 4. Behavioral feature engineering

In [7]:
model_df['on_payment_plan'] = (model_df['pymnt_plan'] == 'y').astype(int)
model_df['entered_hardship'] = (model_df['hardship_flag'] == 'Y').astype(int)

# Combined distress signal -- either event alone is meaningful, both together more so.
model_df['distress_combo'] = model_df['on_payment_plan'] + model_df['entered_hardship']

model_df.drop(columns=['pymnt_plan', 'hardship_flag'], inplace=True)

## 5. Origination-side engineered features

In [8]:
model_df['fico_score_origination'] = (model_df['fico_range_low'] + model_df['fico_range_high']) / 2
model_df.drop(columns=['fico_range_low', 'fico_range_high'], inplace=True)

In [9]:
model_df['credit_card_util_pct'] = model_df['revol_bal'] / (model_df['tot_cur_bal'] + 1)
model_df['installment_income_ratio'] = model_df['installment'] / (model_df['annual_inc'] + 1)
model_df['loan_amount_income_ratio'] = model_df['loan_amnt'] / (model_df['annual_inc'] + 1)

In [10]:
model_df['delinq_flag'] = (model_df['mths_since_last_delinq'] < 12).astype(int)
model_df['bankruptcy_flag'] = (model_df['pub_rec_bankruptcies'] > 0).astype(int)
model_df['historical_delinquency_rate'] = model_df['delinq_2yrs'] / (model_df['total_acc'] + 1)

## 6. Encoding & transformations

In [11]:
model_df = pd.get_dummies(
    model_df, columns=['verification_status', 'home_ownership', 'purpose'], drop_first=True
)

In [12]:
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10
}
model_df['emp_length'] = model_df['emp_length'].map(emp_map)
model_df['term'] = model_df['term'].map({' 36 months': 0, ' 60 months': 1})

In [13]:
model_df = model_df[model_df['dti'] > 0]

In [14]:
# Frequency-encode state: avoids a 50-column one-hot expansion while preserving
# regional concentration signal (also feeds the concentration/HHI metric downstream).
addr_freq = model_df['addr_state'].value_counts(normalize=True)
model_df['addr_state_freq'] = model_df['addr_state'].map(addr_freq)
model_df = model_df.drop(columns=['addr_state'])

In [15]:
# Ordinal-encode sub_grade (A1..G5 -> 1.1..7.5), preserving Lending Club's
# own risk ordering.
subgrade_order = [f"{g}{s}" for g in ['A', 'B', 'C', 'D', 'E', 'F', 'G'] for s in range(1, 6)]
subgrade_float_map = {sg: float(f"{i // 5 + 1}.{i % 5 + 1}") for i, sg in enumerate(subgrade_order)}
model_df['sub_grade'] = model_df['sub_grade'].map(subgrade_float_map)
model_df = model_df.drop(columns=['grade'])  # redundant with sub_grade

In [16]:
model_df['revol_util'] = model_df['revol_util'].clip(upper=100)
model_df['fico_dti_ratio'] = model_df['fico_score_origination'] / (1 + model_df['dti'])

In [17]:
model_df['annual_inc_log'] = np.log1p(model_df['annual_inc'])
model_df['dti_log'] = np.log1p(model_df['dti'])
model_df['revol_bal_log'] = np.log1p(model_df['revol_bal'])
model_df['tot_coll_amt_log'] = np.log1p(model_df['tot_coll_amt'])
model_df['tot_cur_bal_log'] = np.log1p(model_df['tot_cur_bal'])

model_df.drop(columns=['annual_inc', 'dti', 'revol_bal', 'tot_coll_amt', 'tot_cur_bal'], inplace=True)

In [18]:
# Credit history length at origination -- a stability signal, not a leakage risk.
model_df['earliest_cr_line'] = pd.to_datetime(model_df['earliest_cr_line'], format='%b-%Y', errors='coerce')
model_df['issue_d_parsed'] = pd.to_datetime(model_df['issue_d'], format='%b-%Y', errors='coerce')
model_df['account_age_years'] = (model_df['issue_d_parsed'] - model_df['earliest_cr_line']).dt.days / 365.25

model_df.drop(columns=['earliest_cr_line', 'issue_d', 'issue_d_parsed'], inplace=True)

In [19]:
emp_title_counts = model_df['emp_title'].value_counts()
model_df['emp_title_freq'] = model_df['emp_title'].map(emp_title_counts)
model_df.drop(columns=['emp_title'], inplace=True)

## 7. Missing-value flags

In [20]:
for col in ['inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
            'revol_util', 'num_tl_90g_dpd_24m', 'emp_length']:
    model_df[f'{col}_missing'] = model_df[col].isna().astype(int)

## 8. Missing-value imputation

`mths_since_last_delinq`/`mths_since_last_record` NaN is meaningful ("never
happened") rather than random, hence the sentinel fill rather than a mean/mode
fill -- the `_missing` flags above preserve the distinction for the model.
`revol_util` NaN corresponds to borrowers with no revolving accounts (0%
utilization by definition). A small cluster of rows lacking an expanded
credit-report pull is dropped rather than imputed across six correlated fields.

In [21]:
model_df['mths_since_last_delinq'] = model_df['mths_since_last_delinq'].fillna(999)
model_df['mths_since_last_record'] = model_df['mths_since_last_record'].fillna(999)
model_df['revol_util'] = model_df['revol_util'].fillna(0)
model_df['emp_title_freq'] = model_df['emp_title_freq'].fillna(0)
model_df['emp_length'] = model_df['emp_length'].fillna(0)

cluster_cols = ['pct_tl_nvr_dlq', 'mo_sin_old_rev_tl_op', 'tot_coll_amt_log',
                 'tot_cur_bal_log', 'credit_card_util_pct', 'num_tl_90g_dpd_24m']
model_df = model_df.dropna(subset=cluster_cols)
model_df = model_df.dropna(subset=['inq_last_6mths'])

assert model_df.isna().sum().sum() == 0, "Unhandled NaNs remain"
assert not np.isinf(model_df.select_dtypes(include=[np.number])).any().any(), "Inf values remain"

## 9. Save

In [25]:
print(f"Final shape: {model_df.shape}")
print(f"Default rate: {model_df['loan_status'].mean():.3f}")

model_df.to_parquet('../Data/model_df_behavioral.parquet')

Final shape: (1276643, 64)
Default rate: 0.202
